# MTST Stan DDM regression: menu-graph caching vs recomputation

This notebook shows how to:
1. Load an MTST CSV
2. Build menu-graph / caching covariates
3. Fit a DDM regression in Stan (Wiener likelihood)
4. Inspect whether menu distance / menu lag load on **nondecision time** `t0`.


In [ ]:
from pathlib import Path
import pandas as pd
import arviz as az
from cmdstanpy import CmdStanModel

from mtst_covariates import load_mtst_csv, build_subject_covariates


In [ ]:
# --- edit these ---
CSV_PATH = Path('hddm2_fixed_final_5states.csv')
S = 5
SUBJ = 1
STAN_FILE = Path('ddm_cache_regression_subject.stan')

df = load_mtst_csv(CSV_PATH)
sd = build_subject_covariates(df, subj=SUBJ, S=S)

print('N trials:', sd.N)
print('Drift predictors:', sd.drift_names)
print('t0 predictors:', sd.tau_names)
print('a predictors:', sd.a_names)
print('Transition predictors:', sd.trans_names)
print('t0 bounds:', sd.t0_lower, sd.t0_upper)


In [ ]:
stan_data = {
    'N': sd.N,
    'rt': sd.rt.astype(float),
    'choice': sd.choice.astype(int),
    'K_v': sd.x_drift.shape[1],
    'X_v': sd.x_drift.astype(float),
    'K_t0': sd.x_tau.shape[1],
    'X_t0': sd.x_tau.astype(float),
    'K_a': sd.x_a.shape[1],
    'X_a': sd.x_a.astype(float),
    't0_lower': float(sd.t0_lower),
    't0_upper': float(sd.t0_upper),
}


In [ ]:
# Compile + sample (edit sampling args for real runs)
model = CmdStanModel(stan_file=str(STAN_FILE))
fit = model.sample(
    data=stan_data,
    chains=4,
    parallel_chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    adapt_delta=0.95,
    max_treedepth=12,
    seed=2027,
)
idata = az.from_cmdstanpy(posterior=fit, log_likelihood='log_lik')
az.summary(idata, var_names=['b_t0','b_v','b_a'], hdi_prob=0.95)


## Interpretation

- `b_t0[0]` corresponds to the first `t0` predictor (`menu_dist_lag1`):
  positive values mean **farther menu jumps increase nondecision time**.
- `b_t0[1]` corresponds to `log1p_menu_lag`:
  positive values mean **stale menus increase nondecision time**.

These are the cleanest DDM signatures of a caching/retrieval mechanism.
